# Project XYZ — Track B: Qwen3 4B Cybersecurity QLoRA

This notebook trains a specialized cybersecurity adapter for **`Qwen/Qwen3-4B-Instruct-2507`** using **4-bit NF4 QLoRA** on an **NVIDIA T4 GPU (16GB VRAM)** in Google Colab.

### Pipeline Overview
1. **Environment Check**: Verify NVIDIA T4 GPU and CUDA 12.x.
2. **Google Drive Integration**: Mount Drive to persist checkpoints (`MyDrive/ProjectXYZ/qwen3-4b-cyber/`).
3. **Base Model**: Load original Qwen checkpoint in runtime bitsandbytes NF4 4-bit with double quantization.
4. **Assistant-Only Loss**: Non-assistant tokens (system, user, headers) are masked with `-100`.
5. **Baseline Benchmark**: Evaluate un-tuned base model across cybersecurity and regression suites.
6. **QLoRA Training**: Execute supervised fine-tuning with automatic Drive backups and resume capability.
7. **Post-Training Benchmark & Comparison**: Run evaluation suite on trained adapter and compute before/after deltas.

In [ ]:
# CELL 2: GPU Check and CUDA Verification
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable! In VS Code or Colab, connect to an NVIDIA T4 GPU runtime "
        "(Runtime -> Change runtime type -> T4 GPU)."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Connected to GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")

!nvidia-smi


In [23]:
# CELL 3: Install Pinned QLoRA Dependencies
!pip install --upgrade pip
!pip install torch>=2.2.0 transformers>=4.48.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 peft>=0.14.0 trl>=0.12.0 datasets>=2.21.0 safetensors pyyaml scikit-learn

In [24]:
# CELL 4: Print Dependency Versions
import sys
import torch
import transformers
import peft
import trl
import accelerate
import bitsandbytes

print("=" * 50)
print("INSTALLED FRAMEWORK VERSIONS")
print("=" * 50)
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"Accelerate:   {accelerate.__version__}")
print(f"BitsAndBytes: {bitsandbytes.__version__}")
print("=" * 50)

INSTALLED FRAMEWORK VERSIONS
Python:       3.13.15
PyTorch:      2.11.0+cu130 (CUDA: 13.0)
Transformers: 5.20.1
PEFT:         0.21.2
TRL:          1.16.0
Accelerate:   1.15.0
BitsAndBytes: 0.50.2


In [25]:
# CELL 5: Mount and Verify Google Drive
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"Google Drive mount skipped or running in local environment: {e}")
    DRIVE_MOUNTED = False

DRIVE_ROOT = "/content/drive/MyDrive/ProjectXYZ/qwen3-4b-cyber"
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/adapters", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/benchmarks", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/reports", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)

print(f"Google Drive persistence root verified at: {DRIVE_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive persistence root verified at: /content/drive/MyDrive/ProjectXYZ/qwen3-4b-cyber


In [26]:
# CELL 6: Main Configuration Variables
# Adjust hyperparameters for your experiment run if needed

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 1024        # Can be set to 2048 if VRAM permits
LORA_R = 32                  # Rank 32 (fallback 16 for OOM)
LORA_ALPHA = 64              # 2 * rank
LORA_DROPOUT = 0.05
EPOCHS = 3
LEARNING_RATE = 2e-4
MICRO_BATCH_SIZE = 1         # Kept at 1 for T4 safety
GRADIENT_ACCUMULATION = 16   # Effective batch size = 16
RESUME_FROM_CHECKPOINT = None # Set explicit checkpoint path, or None to auto-resume from Drive
DRIVE_ROOT = "/content/drive/MyDrive/ProjectXYZ/qwen3-4b-cyber"

print("Configuration initialized for:", BASE_MODEL)

Configuration initialized for: Qwen/Qwen3-4B-Instruct-2507


In [27]:
# CELL 7: Access / Clone Repository Files
import os, sys
from pathlib import Path

REPO_NAME = "project-xyz"
LOCAL_REPO = Path(f"/content/{REPO_NAME}")

# If running inside Colab remote runtime without workspace clone:
if not LOCAL_REPO.exists() and not Path("src").exists():
    print("Cloning repository into Colab runtime...")
    !git clone -b qwen3-4b-cyber-qlora https://github.com/sajancodes/project-xyz.git /content/project-xyz

if LOCAL_REPO.exists():
    os.chdir(str(LOCAL_REPO))

repo_path = str(Path.cwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Active working directory:", os.getcwd())
assert Path("src").exists(), "src directory not found! Ensure project-xyz files are accessible."

Active working directory: /content/project-xyz


In [28]:
# CELL 8: Validate Dataset
!python scripts/validate_qwen4b_dataset.py


--- Validating: /content/project-xyz/data/qwen4b/processed/train.jsonl ---
Loaded 12 records.
  [PASS] All 12 records conform to schema and quality checks.

--- Validating: /content/project-xyz/data/qwen4b/processed/val.jsonl ---
Loaded 1 records.
  [PASS] All 1 records conform to schema and quality checks.

--- Validating: /content/project-xyz/data/qwen4b/processed/test.jsonl ---
Loaded 1 records.
  [PASS] All 1 records conform to schema and quality checks.

--- Validating: /content/project-xyz/data/qwen4b/benchmarks/cyber_benchmark.jsonl ---
Loaded 2 records.
  [PASS] All 2 records conform to schema and quality checks.

--- Checking Cross-Split Contamination ---
  [PASS] No contamination between Train and Validation.
  [PASS] No contamination between Train and Test.
  [PASS] No contamination between Train and Benchmark.

ALL DATASET VALIDATIONS PASSED SUCCESSFULLY!



In [29]:
# CELL 9: Load Tokenizer
from src.models.qwen4b.loader import load_qwen_tokenizer

tokenizer = load_qwen_tokenizer(BASE_MODEL)
print(f"Official Qwen tokenizer loaded successfully. Vocab size: {len(tokenizer)}")
print(f"Pad token: '{tokenizer.pad_token}', EOS token: '{tokenizer.eos_token}'")

Official Qwen tokenizer loaded successfully. Vocab size: 151669
Pad token: '<|endoftext|>', EOS token: '<|im_end|>'


In [30]:
# CELL 10: Load Qwen 4B in NF4 4-bit
from src.models.qwen4b.loader import load_qwen_model

print(f"Loading {BASE_MODEL} in 4-bit NF4 (fp16 compute dtype)...\n")
model = load_qwen_model(
    model_id_or_path=BASE_MODEL,
    load_in_4bit=True,
    compute_dtype="float16",
    device_map="auto",
    use_cache=False,
    enable_gradient_checkpointing=True,
)
print("Base model successfully loaded and quantized at runtime!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen/Qwen3-4B-Instruct-2507 in 4-bit NF4 (fp16 compute dtype)...



model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Base model successfully loaded and quantized at runtime!


In [31]:
# CELL 11: Attach LoRA Adapter
from src.models.qwen4b.lora import apply_qlora, get_qlora_config

lora_config = get_qlora_config(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = apply_qlora(model, lora_config=lora_config, prepare_kbit=True)
print("LoRA adapter attached successfully!")

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157
LoRA adapter attached successfully!


In [32]:
# CELL 12: Print Model, GPU, and Trainable Parameter Information
from src.models.qwen4b.loader import get_model_parameters_info
from src.training.qwen4b.utils import print_gpu_environment

print_gpu_environment()
param_info = get_model_parameters_info(model)
print(f"Total Parameters:     {param_info['total_parameters']:,}")
print(f"Trainable Parameters: {param_info['trainable_parameters']:,}")
print(f"Trainable Ratio:      {param_info['trainable_percentage']:.4f}%")


HARDWARE & DEPENDENCY ENVIRONMENT
GPU:                  Tesla T4
VRAM:                 14.56 GB
CUDA Available:       True (Version: 13.0)
PyTorch Version:      2.11.0+cu130
Transformers Version: 5.20.1
PEFT Version:         0.21.2
BitsAndBytes Version: 0.50.2
Python Version:       3.13.15

Total Parameters:     2,271,870,464
Trainable Parameters: 66,060,288
Trainable Ratio:      2.9077%


In [33]:
# CELL 13: Tiny QLoRA Forward/Backward Smoke Test (OOM Safety Check)
from src.training.qwen4b.utils import run_smoke_forward_backward_step

success, msg = run_smoke_forward_backward_step(model, tokenizer, max_seq_len=128)
if not success:
    raise RuntimeError(f"Smoke step failed! Review OOM fallback: {msg}")
print(msg)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Smoke step succeeded with initial loss: 5.9961


In [ ]:
# CELL 14: Run BASELINE Benchmark (BEFORE QLoRA)

import logging
from typing import Any, Dict

from src.evaluation.qwen4b.benchmark import run_benchmark

# Show detailed benchmark progress in notebook output
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

print("=" * 70)
print("BASELINE BENCHMARK — Qwen3 4B BEFORE QLoRA")
print("=" * 70)

# Benchmark/inference mode
model.eval()

print("Starting baseline benchmark...")
print("You should see each benchmark question below as it is processed.\n")

base_results: Dict[str, Any] = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    tag="base",
    output_dir=f"{DRIVE_ROOT}/benchmarks",
)

print("\n" + "=" * 70)
print("BASELINE BENCHMARK COMPLETE")
print("=" * 70)

print(f"Questions evaluated: {base_results['num_questions']}")
print(f"Baseline Overall Score: {base_results['overall_score']:.4f}")

print("\nCategory Scores:")
for category, score in base_results["category_scores"].items():
    print(f"  {category:<30} {score:.4f}")

print("\nResults saved to:")
print(f"{DRIVE_ROOT}/benchmarks/base_benchmark_results.json")

print("=" * 70)

2026-10-23 02:21:59,265 | INFO | Running benchmark 'base' on 17 questions...
2026-10-23 02:21:59,267 | INFO | Evaluating [networking_security] cyber_net_01...


BASELINE BENCHMARK — Qwen3 4B BEFORE QLoRA
Starting baseline benchmark...
You should see each benchmark question below as it is processed.



In [ ]:
# CELL 15: Prepare Trainer and Datasets with Assistant-Only Loss Masking
from pathlib import Path
from transformers import Trainer, TrainingArguments
from src.training.qwen4b.checkpointing import DriveBackupCallback, GoogleDriveCheckpointManager
from src.training.qwen4b.collator import DataCollatorForAssistantOnlyLM
from src.training.qwen4b.dataset import CyberChatDataset

drive_mgr = GoogleDriveCheckpointManager(DRIVE_ROOT)
drive_mgr.ensure_directories()

train_dataset = CyberChatDataset(
    data_source="data/qwen4b/processed/train.jsonl",
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)
val_dataset = CyberChatDataset(
    data_source="data/qwen4b/processed/val.jsonl",
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)
collator = DataCollatorForAssistantOnlyLM(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="outputs/qwen3_4b_cyber",
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    num_train_epochs=EPOCHS,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[DriveBackupCallback(drive_mgr)],
)

resume_target = RESUME_FROM_CHECKPOINT or drive_mgr.get_resume_checkpoint(Path("outputs/qwen3_4b_cyber"))
print(f"Trainer successfully initialized. Resume target: {resume_target}")

In [ ]:
# ====================================================================
# ====================================================================
#                 CELL 16: FINAL TRAINING CELL
# ====================================================================
# ====================================================================
# THIS IS THE OFFICIAL QLORA TRAINING CELL.
# Checkpoints automatically persist to Google Drive during training.

print("Starting QLoRA Supervised Fine-Tuning on NVIDIA T4...")
train_result = trainer.train(resume_from_checkpoint=resume_target)
print("=" * 60)
print("QLORA TRAINING STEP COMPLETED SUCCESSFULLY!")
print("=" * 60)
print(train_result)

In [ ]:
# CELL 17: Save Final LoRA Adapter Locally and to Google Drive
local_adapter_dir = Path("outputs/qwen3_4b_cyber/final_adapter")
local_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(local_adapter_dir))
tokenizer.save_pretrained(str(local_adapter_dir))

# Backup to Google Drive
drive_mgr.sync_adapter_to_drive(local_adapter_dir, adapter_name="final_cyber_adapter")
print(f"Adapter saved locally: {local_adapter_dir}")
print(f"Adapter persisted to Drive: {DRIVE_ROOT}/adapters/final_cyber_adapter")

In [ ]:
# CELL 18: Clean Model / GPU State for Clean Evaluation
from src.training.qwen4b.utils import clear_cuda_memory

del trainer
del model
clear_cuda_memory()
print("Training state cleared from GPU memory.")

In [ ]:
# CELL 19: Reload Adapter from Google Drive
from src.models.qwen4b.inference import load_lora_adapter
from src.models.qwen4b.loader import load_qwen_model

print("Loading fresh 4-bit base model...")
eval_base_model = load_qwen_model(BASE_MODEL, load_in_4bit=True, device_map="auto")

adapter_path = f"{DRIVE_ROOT}/adapters/final_cyber_adapter"
print(f"Attaching trained LoRA adapter from Google Drive: {adapter_path}...")
eval_model = load_lora_adapter(eval_base_model, adapter_path)
print("Trained adapter reloaded and verified!")

In [ ]:
# CELL 20: Verify Inference Generation
from src.models.qwen4b.inference import generate_response

prompt = "Explain how SYN flood attacks exploit the TCP three-way handshake and describe two mitigations."
ans = generate_response(eval_model, tokenizer, prompt, max_new_tokens=300)

print("--- INFERENCE VERIFICATION ---")
print("Prompt:", prompt)
print("\nResponse:\n", ans)

In [ ]:
# CELL 21: Run Post-Training Benchmark
from src.evaluation.qwen4b.benchmark import run_benchmark

print("Running post-training benchmark on fine-tuned QLoRA model...")
qlora_results = run_benchmark(
    model=eval_model,
    tokenizer=tokenizer,
    tag="qlora",
    output_dir=f"{DRIVE_ROOT}/benchmarks",
)
print(f"Post-Training Overall Score: {qlora_results['overall_score']:.4f}")
print("Category Scores:", qlora_results["category_scores"])

In [ ]:
# CELL 22: Generate Before-vs-After Comparison Report
from IPython.display import Markdown, display
from src.evaluation.qwen4b.comparison import compare_benchmarks

base_json = f"{DRIVE_ROOT}/benchmarks/base_benchmark_results.json"
report_md = compare_benchmarks(
    base_results=base_json,
    qlora_results=qlora_results,
    output_md_path=f"{DRIVE_ROOT}/reports/comparison_report.md",
)
display(Markdown(report_md))

In [ ]:
# CELL 23: Interactive Cybersecurity Query Session
interactive_query = "How does an attacker exploit an SSRF vulnerability to access the AWS metadata service?" # @param {type:"string"}

response = generate_response(eval_model, tokenizer, interactive_query, max_new_tokens=400)
print("Query:", interactive_query)
print("\nResponse:\n", response)

In [ ]:
# CELL 24: Optional Adapter Merge and 16-bit Export
MERGE_AND_EXPORT = False # @param {type:"boolean"}

if MERGE_AND_EXPORT:
    from src.models.qwen4b.inference import merge_and_export_adapter
    export_target = f"{DRIVE_ROOT}/merged_16bit_model"
    print("Merging adapter into base model (requires ~8GB CPU RAM)...\n")
    merge_and_export_adapter(eval_model, export_target)
    print(f"Merged weights exported to: {export_target}")
else:
    print("Adapter export skipped. Set MERGE_AND_EXPORT=True to export merged weights.")